# Financial Information Retrieval with BM25 and SBERT

*Comparison of lexical and dense retrieval on the BEIR FiQA-2018 benchmark using BM25, Sentence-BERT and FAISS.*

## Project Overview

This notebook compares lexical retrieval with **BM25** and dense retrieval with **SBERT and FAISS** on the **BEIR FiQA-2018 benchmark**. It covers dataset preparation, retrieval, parameter tuning, evaluation, model comparison, and an interactive search interface.

Originally developed as part of a university group project.

### Personal contribution

- Designed the evaluation methodology.
- Performed BM25 parameter optimisation.
- Analysed retrieval performance across models.
- Interpreted experimental results and documented findings.

### Technologies

- Python
- BEIR
- BM25
- Sentence Transformers
- FAISS
- PyTorch
- Gradio

### Contents

- [Project Overview](#project-overview)
- [Environment Setup](#environment-setup)
- [Dataset](#dataset)
- [Data Inspection](#data-inspection)
- [BM25 Baseline](#bm25-baseline)
- [BM25 Parameter Tuning](#bm25-parameter-tuning)
- [Dense Retrieval with SBERT](#dense-retrieval-with-sbert)
- [FAISS Indexing](#faiss-indexing)
- [Evaluation](#evaluation)
- [Model Comparison](#model-comparison)
- [Interactive Search Interface](#interactive-search-interface)
- [Limitations](#limitations)
- [Future Improvements](#future-improvements)
- [References](#references)


## Environment Setup

Install the required packages in the active Python environment before running the notebook. The setup cell uses notebook-compatible `pip` commands and works in hosted environments such as Google Colab as well as compatible local Jupyter environments.

The implementation uses `bm25s` for lexical retrieval. Pyserini was considered initially, but its Anserini binaries required a newer Java runtime than the original execution environment provided. `bm25s` implements Okapi BM25 without an external Java dependency while preserving the same scoring model and tunable parameters.

**Core dependencies:** `beir`, `bm25s`, `faiss-cpu`, `sentence-transformers`, `gradio`, `pandas`, and `torch`.

In [4]:
%%capture
!pip install --upgrade beir
!pip install faiss-cpu bm25s sentence-transformers gradio

import os, logging, warnings, itertools
import pandas as pd

# Suppresses debug output from bm25s
root_logger = logging.getLogger()
root_logger.setLevel(logging.WARNING)

for handler in root_logger.handlers:
    handler.setLevel(logging.WARNING)
logging.getLogger('bm25s').setLevel(logging.WARNING)

# Suppresses HuggingFace Hub and tqdm warnings
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')
warnings.filterwarnings('ignore', category=DeprecationWarning)

## Dataset

FiQA-2018 is a financial question-answering dataset standardized within the BEIR benchmark (Thakur et al., 2021). It contains natural-language financial questions and answer passages collected from financial community platforms. The difference in wording between questions and relevant passages makes it useful for comparing lexical and semantic retrieval.

| Component | Description | Test-set scope |
|:---|:---|:---|
| **Corpus** | Passages available for indexing and retrieval | 57,638 documents |
| **Queries** | Financial questions evaluated in this notebook | 648 queries |
| **Qrels** | Ground-truth relevance mappings for the evaluated queries | 648 query entries |

The dataset is downloaded automatically from the public BEIR repository and loaded through `GenericDataLoader`.

In [5]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

# Downloads and unzips FiQA-2018 from the BEIR repository
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip"
data_path = util.download_and_unzip(url, "/content")

# Load the test split used throughout the experiments
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

print(f"Corpus:  {len(corpus):,} documents")
print(f"Queries: {len(queries):,} queries")
print(f"Qrels:   {len(qrels):,} relevance judgements")

/content/fiqa.zip:   0%|          | 0.00/17.1M [00:00<?, ?iB/s]

  0%|          | 0/57638 [00:00<?, ?it/s]

Corpus:  57,638 documents
Queries: 648 queries
Qrels:   648 relevance judgements


## Data Inspection

Before indexing, the notebook displays one document, one query, and the associated relevance labels. This check confirms that the corpus, queries, and qrels were loaded correctly and illustrates the structure of the retrieval task.

In [6]:
# Sample document, query, and relevance judgement
sample_doc_id = list(corpus.keys())[0]
sample_qid    = list(queries.keys())[0]

print("SAMPLE DOCUMENT")
print(f"  ID:    {sample_doc_id}")
print(f"  Title: {corpus[sample_doc_id].get('title', '(none)')}")
print(f"  Text:  {corpus[sample_doc_id]['text'][:200]}...\n")

print("SAMPLE QUERY")
print(f"  ID:   {sample_qid}")
print(f"  Text: {queries[sample_qid]}\n")

print("RELEVANT DOCUMENTS (qrel)")
for doc_id, score in list(qrels[sample_qid].items())[:3]:
    print(f"  doc_id={doc_id}, relevance={score}")

SAMPLE DOCUMENT
  ID:    3
  Title: 
  Text:  I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems...

SAMPLE QUERY
  ID:   8
  Text: How to deposit a cheque issued to an associate in my business into my business account?

RELEVANT DOCUMENTS (qrel)
  doc_id=566392, relevance=1
  doc_id=65404, relevance=1


## BM25 Baseline

BM25 (*Best Matching 25*) is a lexical retrieval model that ranks a document $d$ for a query $q$ using term frequency, inverse document frequency, and document-length normalization (Robertson & Zaragoza, 2009):

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1\left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

- $f(t,d)$ is the frequency of term $t$ in document $d$.
- $\text{IDF}(t)$ down-weights terms that occur in many documents.
- $|d|$ and $\text{avgdl}$ are the document length and average corpus document length.
- $k_1$ controls term-frequency saturation.
- $b$ controls document-length normalization.

The baseline concatenates each document's title and body, tokenizes the corpus and all 648 test queries without stemming or stop-word removal, retrieves the top 100 documents per query, and evaluates the run with BEIR. Its parameters are $k_1=0.9$ and $b=0.4$.

The stored baseline output reports NDCG@10, MAP@100, Recall@100, and Precision@10.

In [7]:
import bm25s
from beir.retrieval.evaluation import EvaluateRetrieval

# Flatten corpus dictionary into parallel lists of IDs and text strings
# title and body text are concatenated so both fields contribute to BM25 scoring
corpus_ids   = list(corpus.keys())
corpus_texts = [
    (corpus[doc_id].get("title", "") + " " + corpus[doc_id]["text"]).strip()
    for doc_id in corpus_ids
]

# Tokenises corpus and builds BM25 index with default hyperparameters
corpus_tokens = bm25s.tokenize(corpus_texts, stopwords=None, stemmer=None)
retriever     = bm25s.BM25(k1=0.9, b=0.4)
retriever.index(corpus_tokens)

# Tokenises queries using identical settings to the corpus (no stopwords/stemming)
# retrieves the top 100 documents for each of the 648 queries
query_ids    = list(queries.keys())
query_texts  = list(queries.values())
query_tokens = bm25s.tokenize(query_texts, stopwords=None, stemmer=None)
results_idx, scores = retriever.retrieve(query_tokens, k=100)

# Convert raw retrieval output to BEIR format
bm25_results = {
    query_ids[i]: {
        corpus_ids[results_idx[i, j]]: float(scores[i, j])
        for j in range(results_idx.shape[1])
    }
    for i in range(len(query_ids))
}

# Evaluates BM25 retrieval results against ground-truth qrels
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, bm25_results, [10, 100]
)

print("BM25 Baseline  (k1=0.9, b=0.4)\n")
print(f"  NDCG@10:    {ndcg['NDCG@10']:.4f}")
print(f"  MAP@100:    {_map['MAP@100']:.4f}")
print(f"  Recall@100: {recall['Recall@100']:.4f}")
print(f"  P@10:       {precision['P@10']:.4f}")

Split strings:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/57638 [00:00<?, ?it/s]

Split strings:   0%|          | 0/648 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/648 [00:00<?, ?it/s]

BM25 Baseline  (k1=0.9, b=0.4)

  NDCG@10:    0.2345
  MAP@100:    0.1874
  Recall@100: 0.4952
  P@10:       0.0648


The baseline provides a lexical reference point for the later experiments. NDCG@10 measures ranking quality near the top of the list, MAP@100 summarizes precision across relevant results through rank 100, Recall@100 measures the fraction of relevant documents recovered, and P@10 measures the relevant fraction among the first ten results.

Because FiQA relevance labels are sparse, P@10 values are expected to be numerically lower than metrics that aggregate performance across deeper rankings.

## BM25 Parameter Tuning

The notebook evaluates whether BM25 benefits from parameters tailored to FiQA's document lengths and query language. It performs a grid search over:

- $k_1 \in \{0.9, 1.2, 1.5\}$
- $b \in \{0.5, 0.75, 0.9\}$

All nine combinations retrieve 100 documents per query. NDCG@10 is the selection criterion, and MAP@10 is recorded alongside it.

The best configuration is $k_1=0.9$, $b=0.75$. It is re-indexed and evaluated with the full metric set used for the baseline. Because the available experiment tunes and evaluates on the same test split, the resulting comparison should be interpreted with that limitation in mind.

In [8]:
# k1 controls term frequency saturation
# higher values give more weight to repeated terms before the score plateaus
# b controls document length normalisation, higher values penalise longer docs more
tuning_results = []

# NDCG@10 is the selection criterion
# Evaluate all 9 (k1, b) combinations
# for each configuration: rebuild the index, retrieve top-100, evaluate NDCG@10
for k1, b in itertools.product([0.9, 1.2, 1.5], [0.5, 0.75, 0.9]):
    r = bm25s.BM25(k1=k1, b=b)
    r.index(corpus_tokens)
    idx, s = r.retrieve(query_tokens, k=100)
    run = {
        query_ids[i]: {corpus_ids[idx[i,j]]: float(s[i,j]) for j in range(100)}
        for i in range(len(query_ids))
    }

    # Evaluate and record
    n, m, _, _ = EvaluateRetrieval.evaluate(qrels, run, [10])
    tuning_results.append({"k1": k1, "b": b,
                            "NDCG@10": round(n["NDCG@10"], 4),
                            "MAP@10":  round(m["MAP@10"],  4)})
    print(f"  k1={k1}, b={b}:  NDCG@10={n['NDCG@10']:.4f}")

# Sort all configurations by NDCG@10 descending, best config appears at the top
df_tuning = pd.DataFrame(tuning_results).sort_values("NDCG@10", ascending=False)
display(df_tuning)

BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/648 [00:00<?, ?it/s]

  k1=0.9, b=0.5:  NDCG@10=0.2380


BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

...

BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/648 [00:00<?, ?it/s]

  k1=1.5, b=0.9:  NDCG@10=0.2157


,k1,b,NDCG@10,MAP@10
1,0.9,0.75,0.2401,0.1795
3,1.2,0.50,0.2383,0.1780
0,0.9,0.50,0.2380,0.1777
6,1.5,0.50,0.2370,0.1758
4,1.2,0.75,0.2367,0.1758
7,1.5,0.75,0.2321,0.1705
2,0.9,0.90,0.2298,0.1722
5,1.2,0.90,0.2225,0.1653
8,1.5,0.90,0.2157,0.1597


### Best BM25 Configuration

The grid search selects $k_1=0.9$ and $b=0.75$. Relative to the baseline, the larger $b$ applies stronger document-length normalization while the selected $k_1$ leaves term-frequency saturation unchanged.

The gains are modest but consistent across the stored metrics. This tuned configuration is used as the lexical model in the interactive interface and as the strongest BM25 configuration in the final comparison.

In [9]:
# Re-index with the best configuration identified above
best_retriever = bm25s.BM25(k1=0.9, b=0.75)
best_retriever.index(corpus_tokens)
idx, s = best_retriever.retrieve(query_tokens, k=100)

bm25_tuned_results = {
    query_ids[i]: {corpus_ids[idx[i,j]]: float(s[i,j]) for j in range(100)}
    for i in range(len(query_ids))
}

ndcg_t, map_t, recall_t, prec_t = EvaluateRetrieval.evaluate(
    qrels, bm25_tuned_results, [10, 100]
)

print("BM25 Tuned  (k1=0.9, b=0.75)\n")
print(f"  NDCG@10:    {ndcg_t['NDCG@10']:.4f}  (Increase: {ndcg_t['NDCG@10'] - ndcg['NDCG@10']:+.4f})")
print(f"  MAP@100:    {map_t['MAP@100']:.4f}  (Increase: {map_t['MAP@100']  - _map['MAP@100']:+.4f})")
print(f"  Recall@100: {recall_t['Recall@100']:.4f}  (Increase: {recall_t['Recall@100'] - recall['Recall@100']:+.4f})")
print(f"  P@10:       {prec_t['P@10']:.4f}  (Increase: {prec_t['P@10'] - precision['P@10']:+.4f})")

BM25S Count Tokens:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/57638 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/648 [00:00<?, ?it/s]

BM25 Tuned  (k1=0.9, b=0.75)

  NDCG@10:    0.2401  (Increase: +0.0056)
  MAP@100:    0.1918  (Increase: +0.0044)
  Recall@100: 0.5084  (Increase: +0.0132)
  P@10:       0.0674  (Increase: +0.0026)


## Dense Retrieval with SBERT

Lexical retrieval depends on shared terms between a query and a document. In FiQA, relevant financial questions and answers may express the same concept using different vocabulary. Dense retrieval addresses this mismatch by encoding queries and documents into a shared semantic vector space.

The notebook uses `multi-qa-mpnet-base-dot-v1`, a Sentence Transformers model designed for asymmetric question-to-passage retrieval. It produces 768-dimensional embeddings that are L2-normalized before indexing. With normalized vectors, inner-product similarity is equivalent to cosine similarity.

Every document in the 57,638-document corpus is encoded once. The original run used half precision when CUDA was available to reduce GPU memory use. The stored output records an encoding time of approximately four minutes on a T4 GPU.

In [10]:
import torch, gc
from sentence_transformers import SentenceTransformer

# Reduces fragmentation that can cause OOM errors even when total free VRAM is sufficient
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore')  # suppresses HF Hub and model load warnings

# Reclaim available memory before encoding
gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
free_gb = torch.cuda.mem_get_info()[0] / 1024**3 if device == "cuda" else 0
print(f"Device: {device}  |  VRAM free before model load: {free_gb:.1f} GB")

sbert_model = SentenceTransformer("multi-qa-mpnet-base-dot-v1", device=device)

# Use fp16 on CUDA to reduce tensor memory requirements
if device == "cuda":
    sbert_model = sbert_model.half()

free_after = torch.cuda.mem_get_info()[0] / 1024**3 if device == "cuda" else 0
print(f"VRAM free after model load:  {free_after:.1f} GB")

# Encode all corpus documents into 768-dimensional dense vectors
# normalize_embeddings=True applies L2 normalisation so dot-product
# similarity equals cosine similarity as required for FAISS inner-product search
corpus_embed = sbert_model.encode(
    corpus_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(f"Embedding matrix: {corpus_embed.shape}")


Device: cuda  |  VRAM free before model load: 3.5 GB


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VRAM free after model load:  3.0 GB


Batches:   0%|          | 0/1802 [00:00<?, ?it/s]

Embedding matrix: (57638, 768)


## FAISS Indexing

The normalized document embeddings are added to `faiss.IndexFlatIP`, an exact inner-product index. Exact search is appropriate for the 57,638-document corpus and avoids approximation error.

| Property | Detail |
|:---|:---|
| **Index type** | `IndexFlatIP` |
| **Search** | Exact, non-approximate nearest neighbors |
| **Similarity** | Inner product over L2-normalized embeddings |
| **Index size** | 57,638 vectors with 768 dimensions |

The queries are encoded with the same SBERT model and normalization settings, and FAISS retrieves the top 100 documents for each query.

In [11]:
import faiss

# Build an exact FAISS inner-product index
index = faiss.IndexFlatIP(corpus_embed.shape[1])
index.add(corpus_embed)
print(f"FAISS index: {index.ntotal:,} vectors, dimension {corpus_embed.shape[1]}")

# Encode queries with the same model and normalization
query_embed = sbert_model.encode(
    query_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Retrieve the top 100 documents per query
sbert_scores, sbert_idx = index.search(query_embed, k=100)

# Convert FAISS results to BEIR format
sbert_results = {
    query_ids[i]: {
        corpus_ids[sbert_idx[i, j]]: float(sbert_scores[i, j])
        for j in range(100)
    }
    for i in range(len(query_ids))
}

# Evaluate dense retrieval against the qrels
sbert_ndcg, sbert_map, sbert_recall, sbert_prec = EvaluateRetrieval.evaluate(
    qrels, sbert_results, [10, 100]
)

print("\nSBERT Dense Retrieval Results:\n")
print(f"  NDCG@10:    {sbert_ndcg['NDCG@10']:.4f}")
print(f"  MAP@100:    {sbert_map['MAP@100']:.4f}")
print(f"  Recall@100: {sbert_recall['Recall@100']:.4f}")
print(f"  P@10:       {sbert_prec['P@10']:.4f}")

FAISS index: 57,638 vectors, dimension 768


Batches:   0%|          | 0/21 [00:00<?, ?it/s]


SBERT Dense Retrieval Results:

  NDCG@10:    0.4443
  MAP@100:    0.3792
  Recall@100: 0.7936
  P@10:       0.1249


## Evaluation

All retrieval runs are evaluated against the FiQA-2018 test qrels with BEIR's `EvaluateRetrieval` implementation.

| Metric | Interpretation |
|:---|:---|
| **NDCG@10** | Ranking quality within the first ten results, with greater weight on highly ranked relevant documents |
| **MAP@100** | Mean average precision through rank 100 |
| **Recall@100** | Fraction of relevant documents retrieved within the first 100 results |
| **P@10** | Fraction of the first ten results that are relevant |

The same final metric set is used for the baseline BM25, tuned BM25, and SBERT dense runs. The parameter grid additionally records MAP@10.

In [12]:
df_results = pd.DataFrame([
    {"Model":       "BM25 Baseline",
     "k1": 0.9, "b": 0.40,
     "NDCG@10":    round(ndcg['NDCG@10'],        4),
     "MAP@100":    round(_map['MAP@100'],          4),
     "Recall@100": round(recall['Recall@100'],     4),
     "P@10":       round(precision['P@10'],        4)},
    {"Model":       "BM25 Tuned",
     "k1": 0.9, "b": 0.75,
     "NDCG@10":    round(ndcg_t['NDCG@10'],       4),
     "MAP@100":    round(map_t['MAP@100'],          4),
     "Recall@100": round(recall_t['Recall@100'],   4),
     "P@10":       round(prec_t['P@10'],           4)},
    {"Model":       "SBERT Dense",
     "k1": None, "b": None,
     "NDCG@10":    round(sbert_ndcg['NDCG@10'],   4),
     "MAP@100":    round(sbert_map['MAP@100'],      4),
     "Recall@100": round(sbert_recall['Recall@100'],4),
     "P@10":       round(sbert_prec['P@10'],       4)},
])

display(df_results)

,Model,k1,b,NDCG@10,MAP@100,Recall@100,P@10
0,BM25 Baseline,0.9,0.40,0.2345,0.1874,0.4952,0.0648
1,BM25 Tuned,0.9,0.75,0.2401,0.1918,0.5084,0.0674
2,SBERT Dense,NaN,NaN,0.4443,0.3792,0.7936,0.1249


## Model Comparison

The stored results show a small, consistent improvement from BM25 parameter tuning. NDCG@10 increases from 0.2345 to 0.2401, a gain of 0.0056 or 2.4%. The tuned configuration also improves MAP@100, Recall@100, and P@10.

SBERT provides the strongest performance across every reported final metric. Its NDCG@10 of 0.4443 is 84.6% higher than the tuned BM25 value of 0.2401. Recall@100 increases from approximately 0.508 for tuned BM25 to 0.794 for SBERT, showing that semantic retrieval recovers substantially more relevant documents within the first 100 results.

These results indicate that changing the retrieval representation has a much larger effect than tuning BM25 within the lexical paradigm. Dense embeddings help bridge the vocabulary mismatch between conversational financial questions and relevant answer passages.

## Interactive Search Interface

The Gradio interface demonstrates the tuned BM25 and SBERT retrieval models using the indexes and embeddings created earlier in the notebook.

| Feature | Description |
|:---|:---|
| **Query input** | Free-text financial query with submit-on-enter support |
| **Model selector** | Switches between tuned BM25 and SBERT dense retrieval |
| **Top-K slider** | Returns between 1 and 20 results |
| **Results table** | Displays rank, document ID, score, title, and a text snippet |
| **Examples** | Provides representative lexical and semantic queries |

> **Demo availability:** The public `gradio.live` URL shown in the stored output was temporary and is no longer expected to function. Run the interface cell in a compatible environment to create a new session.

In [13]:
import gradio as gr


def search(query: str, model: str, top_k: int) -> pd.DataFrame:
    """Retrieve top-k documents for a query using the selected model."""
    if not query.strip():
        return pd.DataFrame(columns=["Rank", "Doc ID", "Score", "Title", "Snippet"])

    if model == "BM25 (Tuned)":
        tokens         = bm25s.tokenize([query], stopwords=None, stemmer=None)
        idx, scores_q  = best_retriever.retrieve(tokens, k=top_k)
        ranked         = [(corpus_ids[idx[0, j]], float(scores_q[0, j])) for j in range(top_k)]
    else:  # SBERT Dense
        q_embed        = sbert_model.encode([query], normalize_embeddings=True,
                                             convert_to_numpy=True)
        scores_q, idxs = index.search(q_embed, k=top_k)
        ranked         = [(corpus_ids[idxs[0, j]], float(scores_q[0, j])) for j in range(top_k)]

    rows = []
    for rank, (doc_id, score) in enumerate(ranked, 1):
        doc     = corpus.get(doc_id, {})
        title   = (doc.get("title") or "").strip()
        snippet = doc.get("text", "")[:250].replace("\n", " ").strip()
        rows.append({"Rank": rank, "Doc ID": doc_id,
                     "Score": round(score, 4), "Title": title, "Snippet": snippet})

    return pd.DataFrame(rows)


with gr.Blocks(title="FiQA-2018 Search Engine", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """# 🔍 FiQA-2018 Search Engine
Search 57,638 financial documents using BM25 (lexical) or SBERT (semantic) retrieval."""
    )

    with gr.Row():
        query_box = gr.Textbox(
            label="Search Query",
            placeholder="e.g. How do I diversify my investment portfolio?",
            lines=1, scale=4
        )
        model_dd = gr.Dropdown(
            choices=["BM25 (Tuned)", "SBERT Dense"],
            value="SBERT Dense",
            label="Retrieval Model", scale=1
        )
        topk_sl = gr.Slider(1, 20, value=10, step=1, label="Top-K", scale=1)

    search_btn = gr.Button("Search", variant="primary")

    results_df = gr.Dataframe(
        headers=["Rank", "Doc ID", "Score", "Title", "Snippet"],
        label="Retrieved Documents",
        interactive=False,
        wrap=True
    )

    gr.Examples(
        examples=[
            ["How do I diversify my investment portfolio?",       "SBERT Dense",  10],
            ["What is the difference between stocks and bonds?",  "SBERT Dense",  10],
            ["capital gains tax rate long term investments",      "BM25 (Tuned)", 10],
            ["how to open a Roth IRA account",                    "BM25 (Tuned)", 10],
        ],
        inputs=[query_box, model_dd, topk_sl]
    )

    search_btn.click(fn=search, inputs=[query_box, model_dd, topk_sl], outputs=results_df)
    query_box.submit(fn=search, inputs=[query_box, model_dd, topk_sl], outputs=results_df)


demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52c839ab2287685ab5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Limitations

- **Test-split tuning:** BM25 parameters are selected and evaluated on the same FiQA test split, so the tuned result may overestimate generalization performance.
- **No hybrid retrieval:** The notebook compares lexical and dense retrieval independently but does not combine their scores or candidate sets.
- **No SBERT fine-tuning:** The dense retriever uses a pretrained Sentence Transformers model without adaptation to FiQA.
- **No latency benchmarking:** Indexing time and per-query retrieval latency are not measured systematically.
- **No statistical significance testing:** The reported aggregate metric differences are not accompanied by confidence intervals or paired significance tests.

## Future Improvements

- Tune retrieval parameters on a dedicated validation split before final test evaluation.
- Combine BM25 and dense retrieval through score fusion or candidate-set fusion.
- Add a cross-encoder or other reranking stage over the highest-ranked candidates.
- Benchmark indexing time, memory use, and end-to-end query latency.
- Persist BM25 and FAISS indexes so the interactive application can start without rebuilding them.
- Improve reproducibility by pinning dependency versions, recording model revisions, and documenting the execution environment.

## References

Johnson, J., Douze, M. and Jégou, H. (2019) 'Billion-Scale Similarity Search with GPUs', *IEEE Transactions on Big Data*, 7(3), pp. 535–547. Available at: https://doi.org/10.1109/TBDATA.2019.2921572

Lassance, C., Déjean, H. and Clinchant, S. (2024) 'BM25S: Orders of Magnitude Faster Lexical Search via Eager Sparse Scores', *arXiv preprint arXiv:2407.03618*. Available at: https://arxiv.org/abs/2407.03618

Reimers, N. and Gurevych, I. (2019) 'Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks', in *Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing (EMNLP)*. Hong Kong: Association for Computational Linguistics, pp. 3982–3992. Available at: https://doi.org/10.18653/v1/D19-1410

Robertson, S. and Zaragoza, H. (2009) 'The Probabilistic Relevance Framework: BM25 and Beyond', *Foundations and Trends in Information Retrieval*, 3(4), pp. 333–389. Available at: https://doi.org/10.1561/1500000019

Thakur, N., Reimers, N., Rücklé, A., Srivastava, A. and Gurevych, I. (2021) 'BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models', in *Proceedings of the Neural Information Processing Systems Track on Datasets and Benchmarks (NeurIPS Datasets and Benchmarks 2021)*. Available at: https://arxiv.org/abs/2104.08663